# E-Commerce Sales & Customer Analytics

## 01 — Data Understanding

### ## 1. Objective

The objective of this project is to analyze e-commerce sales,
customer behavior, product performance, payment methods,
reviews, and delivery performance.

This notebook focuses on understanding the structure and
quality of the raw datasets before performing data cleaning
and exploratory analysis.

## 2. Dataset and Tools

The project uses multiple related datasets containing
information about:

### Dataset

- Customers
- Orders
- Order items
- Products
- Payments
- Reviews
- Sellers

### Tools

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn

## 3. Load Datasets

In [155]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [217]:
data_path = "../data/raw/"

customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")

In [157]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers
}

dataset_summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

dataset_summary

,Dataset,Rows,Columns
0,Customers,99441,5
1,Orders,99441,8
2,Order Items,112650,7
3,Products,32951,9
4,Payments,103886,5
5,Reviews,99224,7
6,Sellers,3095,4


## 4. Dataset Overview

The dataset consists of multiple relational tables. Each table
represents a different aspect of the e-commerce business.

### 4.1 Dataset Summary


| Table       | Purpose                                |
| ----------- | -------------------------------------- |
| Customers   | Customer location and identification   |
| Orders      | Order status and order lifecycle dates |
| Order Items | Products purchased within each order   |
| Products    | Product attributes and categories      |
| Payments    | Payment methods and transaction values |
| Reviews     | Customer ratings and review comments   |
| Sellers     | Seller location information            |


### 4.2 Sample Records

In [215]:
display(customers.head(3))
display(orders.head(3))
display(order_items.head(3))

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


### 4.3 Data Types

In [216]:
dtype_summary = []

for name, df in datasets.items():
    for column in df.columns:
        dtype_summary.append({
            "Dataset": name,
            "Column": column,
            "Data Type": str(df[column].dtype)
        })

dtype_summary = pd.DataFrame(dtype_summary)

dtype_summary

,Dataset,Column,Data Type
0,Customers,customer_id,str
1,Customers,customer_unique_id,str
2,Customers,customer_zip_code_prefix,int64
3,Customers,customer_city,str
4,Customers,customer_state,str
5,Orders,order_id,str
6,Orders,customer_id,str
7,Orders,order_status,str
8,Orders,order_purchase_timestamp,datetime64[us]
9,Orders,order_approved_at,datetime64[us]


## 5. Data Quality Assessment

### 5.1 Missing Values

In [213]:
missing_values = pd.DataFrame()

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    temp = pd.DataFrame({
        "Dataset": name,
        "Column": missing.index,
        "Missing Values": missing.values,
        "Missing %": (missing.values / len(df) * 100).round(2)
    })

    missing_values = pd.concat(
        [missing_values, temp],
        ignore_index=True
    )

missing_values.sort_values(
    "Missing %",
    ascending=False,
)

,Dataset,Column,Missing Values,Missing %
11,Reviews,review_comment_title,87656,88.34
2,Orders,order_delivered_customer_date,2965,2.98
4,Products,product_name_lenght,610,1.85
3,Products,product_category_name,610,1.85
6,Products,product_photos_qty,610,1.85
5,Products,product_description_lenght,610,1.85
1,Orders,order_delivered_carrier_date,1783,1.79
0,Orders,order_approved_at,160,0.16
7,Products,product_weight_g,2,0.01
8,Products,product_length_cm,2,0.01


### Initial Missing-Value Observations

Several columns contain missing values across the datasets.

- Review comment fields contain a high number of missing values, which may be expected because customers are not required to provide written comments.
- Order delivery-related dates contain missing values and require investigation based on order status.
- Product attributes such as category and descriptive information contain missing values.
- A detailed treatment of these missing values will be performed in the data-cleaning stage.

### 5.2 Duplicate Records

In [218]:
duplicate_summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Duplicate Rows": [
        df.duplicated().sum()
        for df in datasets.values()
    ]
})

duplicate_summary

,Dataset,Duplicate Rows
0,Customers,0
1,Orders,0
2,Order Items,0
3,Products,0
4,Payments,0
5,Reviews,0
6,Sellers,0


### 5.3 Identifier Uniqueness

In [214]:
id_summary = pd.DataFrame({
    "Dataset": [
        "Customers",
        "Orders",
        "Order Items",
        "Products",
        "Payments",
        "Reviews",
        "Sellers"
    ],
    "Rows": [
        len(customers),
        len(orders),
        len(order_items),
        len(products),
        len(payments),
        len(reviews),
        len(sellers)
    ],
    "Unique IDs": [
        customers["customer_id"].nunique(),
        orders["order_id"].nunique(),
        order_items["order_id"].nunique(),
        products["product_id"].nunique(),
        payments["order_id"].nunique(),
        reviews["review_id"].nunique(),
        sellers["seller_id"].nunique()
    ]
})

id_summary

,Dataset,Rows,Unique IDs
0,Customers,99441,99441
1,Orders,99441,99441
2,Order Items,112650,98666
3,Products,32951,32951
4,Payments,103886,99440
5,Reviews,99224,98410
6,Sellers,3095,3095


## 6. Dataset Relationships

The tables are connected through common identifiers.

- `customers.customer_id` → `orders.customer_id`
- `orders.order_id` → `order_items.order_id`
- `orders.order_id` → `payments.order_id`
- `orders.order_id` → `reviews.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`

These relationships will allow the datasets to be combined
for sales, customer, product, payment, review, and delivery
analysis.

## 7. Initial Findings

### Data Quality

- The datasets contain no obvious full-row duplicates based on the initial duplicate checks.
- Several columns contain missing values.
- Missing review comments are common and will require separate treatment from missing transactional data.
- Several order delivery dates are missing and need to be evaluated against order status.
- Product information contains missing category and descriptive attributes.

### Data Type Observations

- Several date fields are currently stored as strings.
- Date fields will need to be converted to datetime before time-based analysis.
- Numeric product and transaction fields should be checked for appropriate data types.

## 8. Next Steps

The next stage of the project is data cleaning and preparation.

The following tasks will be performed:

- Convert date columns to appropriate datetime formats.
- Investigate missing values and determine appropriate treatment.
- Validate identifiers and relationships between tables.
- Check for inconsistent or invalid values.
- Prepare clean datasets for exploratory data analysis.